In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

In [40]:
df = pd.read_csv('base/vendas.csv')
mov = pd.read_csv('base/movestoque.csv')
embalagem = pd.read_csv('base/embalagem.csv')
formapgto = pd.read_csv('base/formapgto.csv')
produto = pd.read_csv('base/produto.csv')
classificacao = pd.read_csv('base/classificacao2.csv')

C:\Users\leopa\AppData\Local\Temp\ipykernel_36856\4291646105.py:1: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('base/vendas.csv')


In [41]:
estoque = mov[['descricao_movimentacao', 'embalagemid',
       'unidadenegocioid', 'datahora','quantidade',
       'estoqueanterior', 'estoque', 'precoreferencial', 'precovenda',
       'customedio']].copy()
estoque = estoque.merge(embalagem[['id', 'produtoid', 'apresentacao', 'codigobarras', 
                         'markup', 'descricao']],
                         left_on='embalagemid', right_on='id', how='left')
estoque = estoque.merge(classificacao,
                        on='produtoid',
                        how='left'
                        )
estoque.rename(columns={'nome': 'classificacao_nome'}, inplace=True)

mapping = {
    93733: "DROGARIA JB - F01 - MATRIZ",
    93729: "DROGARIA JB - F02 - CAMERINO",
    93730: "DROGARIA JB - F03 - AILKA"
}

estoque["filial_nome"] = estoque["unidadenegocioid"].map(mapping)

mapping2 = {
    93733: 1,
    93729: 2,
    93730: 3
} 
estoque["filial_codigo"] = estoque["unidadenegocioid"].map(mapping2)
estoque = estoque[estoque['unidadenegocioid'].isin(mapping.keys())]

In [ ]:
estoque

In [30]:
estoque.columns

Index(['descricao_movimentacao', 'embalagemid', 'unidadenegocioid', 'datahora',
       'quantidade', 'estoqueanterior', 'estoque', 'precoreferencial',
       'precovenda', 'customedio', 'id', 'produtoid', 'apresentacao',
       'codigobarras', 'markup', 'descricao', 'classificacaoid',
       'classificacao_nome', 'caminho', 'filial_nome', 'filial_codigo'],
      dtype='object')

In [42]:
def extrair_class_painome(caminho):
    if caminho is None or pd.isna(caminho):
        return "Sem classificação"
    if 'QUERODELIVERY' in caminho:
        return 'QUERODELIVERY'
    partes = caminho.split(' > ')
    return partes[1] if len(partes) > 1 else None

estoque['class_painome'] = estoque['caminho'].apply(extrair_class_painome)

In [48]:
class_estoque = ['VAREJO', 'PRESCRIÇÃO', 'INDICAÇÃO', 'SBB', 
       'USO CONSUMO E SERVIÇOS']

In [ ]:
estoque = estoque[estoque['class_painome'].isin(class_estoque)]

In [50]:
estoque[estoque['datahora']>='2025-03-16'].shape

(283217, 22)

In [51]:
estoque.columns

Index(['descricao_movimentacao', 'embalagemid', 'unidadenegocioid', 'datahora',
       'quantidade', 'estoqueanterior', 'estoque', 'precoreferencial',
       'precovenda', 'customedio', 'id', 'produtoid', 'apresentacao',
       'codigobarras', 'markup', 'descricao', 'classificacaoid',
       'classificacao_nome', 'caminho', 'filial_nome', 'filial_codigo',
       'class_painome'],
      dtype='object')

In [52]:
estoque['descricao_movimentacao'].unique()

array(['Venda', 'Transferência', 'Recebimento Físico',
       'Recebimento Transferência', 'Baixa de Estoque',
       'Conversão Embalagem Saída', 'Conversão Embalagem Entrada',
       'Devolução de Venda', 'Inventário Saída', 'Inventário Entrada',
       'Estorno de Venda', 'Devolução de Compra',
       'Estorno de Item da Transferência',
       'Estorno de Item de Recebimento Físico',
       'Estorno de Item de Recebimento de Transferência'], dtype=object)

In [20]:
movinicial = pd.read_csv('base/movinicial.csv')

In [23]:
pd.concat([mov, movinicial]).to_csv('base/movestoque.csv', index=False)

In [11]:
df_atualiza = pd.read_csv('base/venda0606.csv')

In [14]:
pd.concat([df[df['venda_datahorafechamento']<='2025-06-06'], df_atualiza]).to_csv('base/vendas.csv', index=False)

In [4]:
df['venda_datahorafechamento'].max()

'2025-06-06 14:23:31'

In [5]:
mov['datahora'].max()

'2025-06-06 14:31:41'

In [9]:
df['venda_datahorafechamento'] = pd.to_datetime(df['venda_datahorafechamento'], format='%Y-%m-%d %H:%M:%S')
df['data_venda_apenas'] = df['venda_datahorafechamento'].dt.date

DATA_ATUAL_SIMULADA = datetime(2025, 6, 4).date()


In [18]:
mov['datahora'].min()

'2025-04-01 01:44:10'

In [51]:
df[(df['embalagem_codigobarras']==7899941203273)&
   df['filial_codigo']==1][['embalagem_id', 'embalagem_codigobarras',
       'embalagem_etiqueta','produto_codigo',]]

,embalagem_id,embalagem_codigobarras,embalagem_etiqueta,produto_codigo
15025,376664,7.899941e+12,31525,29915
15026,376664,7.899941e+12,31525,29915
50367,376664,7.899941e+12,31525,29915
50368,376664,7.899941e+12,31525,29915
50369,376664,7.899941e+12,31525,29915
50370,376664,7.899941e+12,31525,29915
82199,376664,7.899941e+12,31525,29915
82200,376664,7.899941e+12,31525,29915
113478,376664,7.899941e+12,31525,29915
113479,376664,7.899941e+12,31525,29915


In [14]:
df_vendas = df.copy()

In [24]:
mes_atual = DATA_ATUAL_SIMULADA.month
ano_atual = DATA_ATUAL_SIMULADA.year
dia_atual_simulado = DATA_ATUAL_SIMULADA.day
data_dia_anterior = DATA_ATUAL_SIMULADA - timedelta(days=1)

In [23]:
vendas_mes_atual_acumulada = df_vendas[
        (df_vendas['data_venda_apenas'].apply(lambda x: x.month if pd.notnull(x) and hasattr(x, 'month') else -1) == mes_atual) &
        (df_vendas['data_venda_apenas'].apply(lambda x: x.year if pd.notnull(x) and hasattr(x, 'year') else -1) == ano_atual) &
        (df_vendas['data_venda_apenas'].apply(lambda x: x.day if pd.notnull(x) and hasattr(x, 'day') else -1) <= dia_atual_simulado)
    ]
vendas_mes_atual_acumulada['item_valortotal'].sum()

np.float64(231128.36000000002)

In [25]:
vendas_dia_anterior = df_vendas[df_vendas['data_venda_apenas'] == data_dia_anterior]
vendas_dia_anterior['item_valortotal'].sum()

np.float64(72150.97)

In [13]:
df[df['data_venda_apenas']==DATA_ATUAL_SIMULADA - timedelta(days=1)].groupby('filial_codigo')['item_valortotal'].sum()

filial_codigo
1    31092.97
2    17357.60
3    23700.40
Name: item_valortotal, dtype: float64

In [35]:
mov = pd.read_csv('base/movestoque.csv')
curvaabc = pd.read_csv('base/curvaabc.csv')
produto = pd.read_csv('base/produto.csv')
embalagem = pd.read_csv('base/embalagem.csv')
classificacao = pd.read_csv('base/classificacao2.csv')

estoque = mov[['descricao_movimentacao', 'embalagemid',
       'unidadenegocioid', 'datahora','quantidade',
       'estoqueanterior', 'estoque', 'precoreferencial', 'precovenda',
       'customedio']].copy()
estoque = estoque.merge(embalagem[['id', 'produtoid', 'apresentacao', 'codigobarras', 
                         'markup', 'descricao']],
                         left_on='embalagemid', right_on='id', how='left')
estoque = estoque.merge(curvaabc[['produtoid', 'unidadenegocioid','nome']],
                            left_on=['produtoid', 'unidadenegocioid'], 
                            right_on=['produtoid', 'unidadenegocioid'], 
                            how='left')
estoque.rename(columns={'nome': 'curvaabc',
                        }, inplace=True)
estoque = estoque.merge(classificacao,
                        on='produtoid',
                        how='left'
                        )
estoque.rename(columns={'nome': 'classificacao_nome'}, inplace=True)

mapping = {
    93733: "DROGARIA JB - F01 - MATRIZ",
    93729: "DROGARIA JB - F02 - CAMERINO",
    93730: "DROGARIA JB - F03 - AILKA"
}

estoque["filial_nome"] = estoque["unidadenegocioid"].map(mapping)

mapping2 = {
    93733: 1,
    93729: 2,
    93730: 3
} 
estoque["filial_codigo"] = estoque["unidadenegocioid"].map(mapping2)
estoque = estoque[estoque['unidadenegocioid'].isin(mapping.keys())]

In [36]:
estoque.columns

Index(['descricao_movimentacao', 'embalagemid', 'unidadenegocioid', 'datahora',
       'quantidade', 'estoqueanterior', 'estoque', 'precoreferencial',
       'precovenda', 'customedio', 'id', 'produtoid', 'apresentacao',
       'codigobarras', 'markup', 'descricao', 'curvaabc', 'classificacaoid',
       'classificacao_nome', 'caminho', 'filial_nome', 'filial_codigo'],
      dtype='object')

In [48]:
estoque[(estoque['codigobarras']==7899941203273)&
        (estoque['filial_codigo']==1)&
        (estoque['descricao_movimentacao']=='Venda')][['datahora',
       'quantidade','precoreferencial',
       'precovenda', 'customedio','curvaabc', ]]

,datahora,quantidade,precoreferencial,precovenda,customedio,curvaabc
417881,2025-05-28 13:48:29,-1.0,82.98,141.07,85.96,B
417882,2025-05-28 13:48:29,-1.0,82.98,141.07,85.96,B
417885,2025-05-28 13:48:29,-1.0,82.98,141.07,85.96,B
417886,2025-05-28 13:48:29,-1.0,82.98,141.07,85.96,B
420373,2025-05-30 11:38:57,-1.0,82.98,141.07,85.96,B
420374,2025-05-30 11:38:57,-1.0,82.98,141.07,85.96,B


In [38]:
saldao = pd.read_csv('base/saldao.csv', sep=';', decimal=',')

In [41]:
saldao = saldao.merge(estoque[estoque['filial_codigo'].isin([1,2,3])], left_on=['Un. Neg.','Cód. Barras/Etiq.'], right_on=['filial_codigo','codigobarras'], how='left', suffixes=('', '_estoque'))
saldao[saldao['embalagemid'].isna()]

,Un. Neg.,Cód. Barras/Etiq.,Embalagem,P. Oferta,N_Preço Ajustado,descricao_movimentacao,embalagemid,unidadenegocioid,datahora,quantidade,...,apresentacao_estoque,codigobarras_estoque,markup_estoque,descricao_estoque,curvaabc_estoque,classificacaoid_estoque,classificacao_nome_estoque,caminho_estoque,filial_nome_estoque,filial_codigo_estoque
0,1,7898953484854,CORTADOR DE COMPRIMIDOS CRISTAL,5.98,4.99,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
65,1,7891158106538,PEDIASURE 400G BAUNILHA,72.58,50.99,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
70,1,7891158106545,PEDIASURE 400G MORANGO,68.42,47.99,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
71,1,7891158106576,PEDIASURE 850G MORANGO,127.89,89.99,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
102,1,7896046701581,SECANTE DE ESMALTE LISS 400ML SPRAY,18.21,10.99,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10111,13,7898636191833,SH+COND TIO NACHO 415ML CLAREADOR,65.79,39.99,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10112,13,7897217403273,ESC DENTAL JADEPRO REF 032 POPMAX DURA,4.62,2.99,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10113,13,7899706188302,SH+COND ELSEVE 375+170ML HIDRA HIALU,32.17,19.99,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10114,13,7891150034952,SAB REXONA 84G ANTIBACTERIANO FRESH,1.99,1.99,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [37]:
estoque

,descricao_movimentacao,embalagemid,unidadenegocioid,datahora,quantidade,estoqueanterior,estoque,precoreferencial,precovenda,customedio,...,apresentacao,codigobarras,markup,descricao,curvaabc,classificacaoid,classificacao_nome,caminho,filial_nome,filial_codigo
0,Venda,1023453,93730,2025-04-25 08:21:41,-1.0,87.0,86.0,1.89,2.64,1.8457,...,NaN,7.895800e+12,39.6825,TRIDENT 8G MELANCIA,B,103465.0,VAREJO LIBERADOS,PRINCIPAL > VAREJO > VAREJO LIBERADOS,DROGARIA JB - F03 - AILKA,3.0
8,Venda,380886,93730,2025-04-28 20:00:49,-1.0,3.0,2.0,51.01,70.52,47.2760,...,NaN,7.894917e+12,38.2474,"CONCARDIO 2,5 MG C/30 CPR",B,103415.0,REFERENCIA OL,PRINCIPAL > PRESCRIÇÃO > PRODUTOS OL > REFEREN...,DROGARIA JB - F03 - AILKA,3.0
15,Venda,419888,93733,2025-04-25 08:22:49,-1.0,6.0,5.0,18.73,26.22,18.8246,...,NaN,7.891011e+12,39.9893,PROT DIAR CAREFREE LV 80 PG 60,B,103477.0,ABSORV/ALGODAO,PRINCIPAL > VAREJO > ABSORV/ALGODAO,DROGARIA JB - F01 - MATRIZ,1.0
17,Transferência,977681,93730,2025-04-25 08:54:06,-1.0,3.0,2.0,81.36,109.14,79.3960,...,NaN,7.896637e+12,34.1445,RAHIME 8 MG C/30 CPR,C,103434.0,REFERENCIA OL - PSICOTROPICO,PRINCIPAL > PRESCRIÇÃO > PRODUTOS OL > REFEREN...,DROGARIA JB - F03 - AILKA,3.0
20,Venda,853697,93729,2025-04-25 08:22:14,-1.0,2.0,1.0,74.64,100.13,66.6843,...,NaN,7.896112e+12,34.1506,ATIVB 1000 MCG C/30 CPR,B,103415.0,REFERENCIA OL,PRINCIPAL > PRESCRIÇÃO > PRODUTOS OL > REFEREN...,DROGARIA JB - F02 - CAMERINO,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
432117,Venda,1055815,93733,2025-06-06 14:22:58,-1.0,477.0,476.0,7.51,10.38,0.9463,...,NaN,7.896112e+12,38.2157,LOSARTANA POTASSICA 50 MG C/30 CPR(TEU),A,5534686.0,GENERICO FARMACIA POPULAR,PRINCIPAL > INDICAÇÃO > GENÉRICOS > GENERICO F...,DROGARIA JB - F01 - MATRIZ,1.0
432118,Venda,227764,93733,2025-06-06 14:29:13,-1.0,4.0,3.0,3.25,4.55,2.7433,...,NaN,7.891182e+12,40.0000,ESMALTE RISQUE 8ML RENDA,C,103485.0,ESMALTES,PRINCIPAL > SBB > ESMALTES,DROGARIA JB - F01 - MATRIZ,1.0
432119,Venda,10840638,93733,2025-06-06 14:29:13,-1.0,4.0,3.0,2.57,3.59,2.5700,...,NaN,7.896112e+12,39.6887,"ESMALTE IMPALA 7,5ML LEMBRANCAS",C,103485.0,ESMALTES,PRINCIPAL > SBB > ESMALTES,DROGARIA JB - F01 - MATRIZ,1.0
432120,Venda,1079676,93733,2025-06-06 14:30:27,-1.0,39.0,38.0,97.12,134.26,3.2004,...,NaN,7.897596e+12,38.2414,SINVASTATINA 20 MG C/30 CPR(SAN),A,5534686.0,GENERICO FARMACIA POPULAR,PRINCIPAL > INDICAÇÃO > GENÉRICOS > GENERICO F...,DROGARIA JB - F01 - MATRIZ,1.0


In [34]:
estoque_saldao = estoque[estoque['codigobarras'].isin(lista_saldao)]

In [35]:
print(estoque_saldao.shape[0], "itens encontrados no estoque com código de barras do saldao")
print(estoque.shape[0], "itens totais no estoque")

10838 itens encontrados no estoque com código de barras do saldao
204651 itens totais no estoque


In [59]:
classificacao = pd.read_csv('base/classificacao2.csv')

In [61]:
classificacao

,produtoid,classificacaoid,nome,caminho
0,103501,103465,VAREJO LIBERADOS,PRINCIPAL > VAREJO > VAREJO LIBERADOS
1,103558,103485,ESMALTES,PRINCIPAL > SBB > ESMALTES
2,103708,103465,VAREJO LIBERADOS,PRINCIPAL > VAREJO > VAREJO LIBERADOS
3,103804,103420,SIMILAR,PRINCIPAL > INDICAÇÃO > SIMILARES > SIMILAR
4,103860,103420,SIMILAR,PRINCIPAL > INDICAÇÃO > SIMILARES > SIMILAR
...,...,...,...,...
19986,1107160,103464,VITAMINA,PRINCIPAL > INDICAÇÃO > VITAMINAS > VITAMINA
19987,128117,103464,VITAMINA,PRINCIPAL > INDICAÇÃO > VITAMINAS > VITAMINA
19988,128173,103464,VITAMINA,PRINCIPAL > INDICAÇÃO > VITAMINAS > VITAMINA
19989,908304,103417,PRODUTO LIBERADO,PRINCIPAL > PRESCRIÇÃO > PRODUTOS LIBERADOS > ...


In [45]:
mov = pd.read_csv('base/movestoque.csv')
curvaabc = pd.read_csv('base/curvaabc.csv')
laboratorio = pd.read_csv('base/laboratorio.csv')
produto = pd.read_csv('base/produto.csv')

In [65]:
estoque = mov[['descricao_movimentacao', 'embalagemid',
       'unidadenegocioid', 'datahora','quantidade',
       'estoqueanterior', 'estoque', 'precoreferencial', 'precovenda',
       'customedio']].copy()
estoque = estoque.merge(embalagem[['id', 'produtoid', 'apresentacao', 'codigobarras', 
                         'markup', 'descricao']],
                         left_on='embalagemid', right_on='id', how='left')
estoque = estoque.merge(curvaabc[['produtoid', 'unidadenegocioid','nome']],
                            left_on=['produtoid', 'unidadenegocioid'], 
                            right_on=['produtoid', 'unidadenegocioid'], 
                            how='left')
estoque.rename(columns={'nome': 'curvaabc',
                        'id':'embalagemid'}, inplace=True)
estoque = estoque.merge(classificacao,
                        on='produtoid',
                        how='left'
                        )
estoque.rename(columns={'nome': 'classificacao_nome'}, inplace=True)

In [73]:
mapping = {
    93733: "DROGARIA JB - F01 - MATRIZ",
    93729: "DROGARIA JB - F02 - CAMERINO",
    93730: "DROGARIA JB - F03 - AILKA"
}

estoque["filial_nome"] = estoque["unidadenegocioid"].map(mapping)


In [74]:
estoque

,descricao_movimentacao,embalagemid,unidadenegocioid,datahora,quantidade,estoqueanterior,estoque,precoreferencial,precovenda,customedio,...,produtoid,apresentacao,codigobarras,markup,descricao,curvaabc,classificacaoid,classificacao_nome,caminho,filial_nome
0,Venda,1023395,93730,2025-01-01 20:39:58,-1.0,127.0,126.0,0.30,0.42,0.3000,...,1023392.0,NaN,7.893261e+07,40.0000,GOMA DE MASCAR CHICLETS 2.8G HORTELA,C,103465.0,VAREJO LIBERADOS,PRINCIPAL > VAREJO > VAREJO LIBERADOS,DROGARIA JB - F03 - AILKA
1,Venda,1023395,93730,2025-01-01 20:44:13,-1.0,126.0,125.0,0.30,0.42,0.3000,...,1023392.0,NaN,7.893261e+07,40.0000,GOMA DE MASCAR CHICLETS 2.8G HORTELA,C,103465.0,VAREJO LIBERADOS,PRINCIPAL > VAREJO > VAREJO LIBERADOS,DROGARIA JB - F03 - AILKA
2,Venda,1023395,93730,2025-01-01 20:44:13,-1.0,125.0,124.0,0.30,0.42,0.3000,...,1023392.0,NaN,7.893261e+07,40.0000,GOMA DE MASCAR CHICLETS 2.8G HORTELA,C,103465.0,VAREJO LIBERADOS,PRINCIPAL > VAREJO > VAREJO LIBERADOS,DROGARIA JB - F03 - AILKA
3,Venda,1014074,93730,2025-01-01 20:49:17,-1.0,27.0,26.0,9.32,18.61,5.8100,...,1014071.0,NaN,7.897948e+12,150.0861,RESSALIV 269ML GAS CITRUS,D,7413829.0,SIMILAR SBB,PRINCIPAL > INDICAÇÃO > SIMILARES > SIMILAR SBB,DROGARIA JB - F03 - AILKA
4,Venda,862322,93733,2025-01-02 07:03:07,-1.0,27.0,26.0,16.56,64.90,19.9900,...,862319.0,NaN,7.898650e+12,250.4320,VITASANUS MAGNESIO DIMALA C/60 CAPS,B,103464.0,VITAMINA,PRINCIPAL > INDICAÇÃO > VITAMINAS > VITAMINA,DROGARIA JB - F01 - MATRIZ
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
442725,Venda,1043613,93730,2025-05-30 21:12:27,-1.0,20.0,19.0,24.58,32.97,6.0838,...,1043610.0,NaN,7.896005e+12,34.1334,CETOCONAZOL CR C/30 GR(EMS),B,103426.0,GENERICO,PRINCIPAL > INDICAÇÃO > GENÉRICOS > GENERICO,DROGARIA JB - F03 - AILKA
442726,Venda,139134,93730,2025-05-30 21:22:32,-1.0,8.0,7.0,10.51,14.71,10.0680,...,139131.0,NaN,7.891010e+12,39.9619,PROT DIAR CAREFREE C/40 BRISA S/PERF,C,103477.0,ABSORV/ALGODAO,PRINCIPAL > VAREJO > ABSORV/ALGODAO,DROGARIA JB - F03 - AILKA
442727,Venda,821879,93730,2025-05-30 21:27:45,-1.0,22.0,21.0,17.81,23.89,3.8993,...,821876.0,NaN,7.896112e+12,34.1381,IBUPRIL 400 MG C/8 CAPS,B,103420.0,SIMILAR,PRINCIPAL > INDICAÇÃO > SIMILARES > SIMILAR,DROGARIA JB - F03 - AILKA
442728,Venda,1026349,93730,2025-05-30 21:43:21,-1.0,123.0,122.0,0.31,0.43,0.3083,...,1026346.0,NaN,7.895800e+12,38.7097,GOMA DE MASCAR BUBBALOO 5G MORANGO,C,103465.0,VAREJO LIBERADOS,PRINCIPAL > VAREJO > VAREJO LIBERADOS,DROGARIA JB - F03 - AILKA


In [17]:
curvaabc

,id,produtoid,unidadenegocioid,curvaabcvalorid,curvaabcquantidadeid,datahoraalteracao,id-2,nome,porcentagem,intervaloinicial,intervalofinal
0,64111,545507,93729,343,343,NaN,343,D,0,100,100
1,23659,103558,93722,343,343,NaN,343,D,0,100,100
2,50044,706046,93733,342,342,2025-05-21 00:00:14,342,C,20,80,100
3,64979,601233,93729,343,343,NaN,343,D,0,100,100
4,23664,104012,93722,343,343,NaN,343,D,0,100,100
...,...,...,...,...,...,...,...,...,...,...,...
259761,213295,493049,93726,341,342,2025-06-01 00:00:24,341,B,30,50,80
259762,219296,862039,93726,340,341,2025-06-01 00:00:24,340,A,50,0,50
259763,218973,842564,93726,341,341,2025-06-01 00:00:24,341,B,30,50,80
259764,211261,354293,93726,341,342,2025-06-01 00:00:24,341,B,30,50,80


In [9]:
mov[['descricao_movimentacao', 'embalagemid',
       'unidadenegocioid', 'datahora','quantidade',
       'estoqueanterior', 'estoque', 'precoreferencial', 'precovenda',
       'customedio']]['descricao_movimentacao'].unique()

array(['Venda', 'Transferência', 'Recebimento Físico',
       'Recebimento Transferência',
       'Estorno de Item de Recebimento Físico',
       'Conversão Embalagem Saída', 'Conversão Embalagem Entrada',
       'Baixa de Estoque', 'Estorno de Venda', 'Devolução de Compra',
       'Devolução de Venda', 'Inventário Saída', 'Inventário Entrada',
       'Estorno de Item da Transferência', 'Estorno Devolução Compra',
       'Estorno de Item de Recebimento de Transferência'], dtype=object)

In [75]:
df.columns

Index(['filial_codigo', 'filial_nome', 'filial_cnpj', 'filial_cidade',
       'caixa_numero', 'caixa_unidnegocioid', 'caixa_descricao', 'nome_caixa',
       'venda_status', 'venda_coo', 'orcamento_codigo', 'orcamento_tipo',
       'orcamento_status', 'orcamento_datahora', 'orcamento_identificacao',
       'orcamento_usuarioid', 'orcamento_unidadenegocioid',
       'orcamento_formapagamentoid', 'orcamento_pessoaid',
       'orcamento_cpfcnpjconsumidor', 'orcamento_vendaid', 'nome_vendedor',
       'valor_total', 'venda_usuariolibcliente', 'venda_usuariolibpagatrasado',
       'venda_libusuariocancelamento', 'venda_tipodocfiscal',
       'venda_datahoraabertura', 'venda_datahorafechamento', 'item_sequencia',
       'item_status', 'item_tipoaliquota', 'item_quantidade',
       'item_valorunitario', 'item_origemdesconto', 'item_tipodesconto',
       'item_desconto', 'item_valortotal', 'item_cadernoofertaid',
       'item_valordesconto', 'item_movimentacaoestoque',
       'item_diasparavenc

In [40]:
df = df.merge(formapgto[['id','nome']], left_on='orcamento_formapagamentoid', right_on='id', how='left')

In [42]:
df['venda_datahorafechamento'] = pd.to_datetime(df['venda_datahorafechamento'], format='%Y-%m-%d %H:%M:%S')

In [3]:
mov_filtrado = mov.loc[mov.groupby(['embalagemid', 'unidadenegocioid'])['datahora'].idxmax()]
mov_filtrado.reset_index(drop=True, inplace=True)
item_custo = mov_filtrado[['embalagemid',
       'unidadenegocioid', 'customedio','datahora']].merge(embalagem[['id','codigobarras']], left_on='embalagemid', right_on='id', how= 'left')

In [51]:
df = df.merge(mov_filtrado[['embalagemid',
       'unidadenegocioid', 'customedio']], left_on=['embalagem_id', 'caixa_unidnegocioid'], right_on=['embalagemid', 'unidadenegocioid'], how='left')

In [55]:
df['customedio']  =df['customedio'].fillna(0)

In [69]:
DATA_ATUAL_SIMULADA = datetime(2025, 5, 30).date()
# Converter DATA_ATUAL_SIMULADA para pd.Timestamp para os cálculos:
data_fim = pd.Timestamp(DATA_ATUAL_SIMULADA) - pd.Timedelta(days=1)
data_inicio = data_fim - pd.Timedelta(days=90)

# Converter a coluna de data/hora, se ainda não estiver em datetime
df['venda_datahorafechamento'] = pd.to_datetime(df['venda_datahorafechamento'], format='%Y-%m-%d %H:%M:%S')

# Filtrar os últimos 90 dias (até hoje - 1)
df_90 = df[(df['venda_datahorafechamento'] >= data_inicio) & (df['venda_datahorafechamento'] <= data_fim)].copy()

# Extrair a hora de venda
df_90['hora'] = df_90['venda_datahorafechamento'].dt.hour

# Agrupar por filial e hora, calculando o ticket médio = total de item_valortotal / número de vendas únicas (venda_coo)
ticket_por_hora = (
    df_90.groupby(['filial_nome', 'hora'])
         .agg(
             total_valor=('item_valortotal', 'sum'),
             n_vendas=('venda_coo', 'nunique')
         )
         .reset_index()
)

ticket_por_hora['ticket_medio'] = ticket_por_hora['total_valor'] / ticket_por_hora['n_vendas']



In [78]:
def calcular_metas_vendedores(df, df_ticket_por_hora):
    """
    Calcula para cada vendedor:
      - filial_nome
      - nome_vendedor
      - Número total de vendas únicas da loja (venda_coo) 
      - Número de vendas únicas do vendedor
      - Ticket médio geral da loja  
      - Ticket médio das horas em que o vendedor tem vendas  
      - Meta do vendedor = (vendas_vendedor / vendas_loja) * (ticket_medio_vendedor / ticket_medio_loja)
    Retorna um DataFrame com esses dados.
    """
    import numpy as np

    resultados = []
    
    # Itera sobre cada vendedor único no DataFrame
    for vendedor in df['nome_vendedor'].unique():
        df_vend = df[df['nome_vendedor'] == vendedor]
        if df_vend.empty:
            continue
        # Supõe que o vendedor atua em uma única loja
        loja = df_vend['filial_nome'].iloc[0]
        
        # Total de vendas da loja (venda_coo únicos)
        vendas_loja = df[df['filial_nome'] == loja]['venda_coo'].nunique()
        # Vendas do vendedor (venda_coo únicos)
        vendas_vend = df_vend['venda_coo'].nunique()
        proporcao_vendas = vendas_vend / vendas_loja if vendas_loja > 0 else np.nan
        
        # Horas em que o vendedor tem vendas
        horas_vendedor = df_vend['hora'].unique()
        
        # Ticket médio da loja apenas nas horas de atuação do vendedor
        df_ticket_vend = df_ticket_por_hora[
            (df_ticket_por_hora['filial_nome'] == loja) & 
            (df_ticket_por_hora['hora'].isin(horas_vendedor))
        ]
        ticket_medio_vend = df_ticket_vend['ticket_medio'].mean() if not df_ticket_vend.empty else np.nan
        
        # Ticket médio geral da loja (todas as horas)
        df_ticket_loja = df_ticket_por_hora[df_ticket_por_hora['filial_nome'] == loja]
        ticket_medio_loja = df_ticket_loja['ticket_medio'].mean() if not df_ticket_loja.empty else np.nan
        
        # Ponderação: razão entre o ticket médio nas horas do vendedor e o geral da loja
        ponderacao = ticket_medio_vend / ticket_medio_loja if (ticket_medio_loja and ticket_medio_loja > 0) else np.nan
        
        # Meta do vendedor: produto da proporção de vendas pela ponderação
        meta = proporcao_vendas * ponderacao if (not np.isnan(proporcao_vendas) and not np.isnan(ponderacao)) else np.nan
        
        resultados.append({
            "filial_nome": loja,
            "nome_vendedor": vendedor,
            "vendas_loja": vendas_loja,
            "vendas_vendedor": vendas_vend,
            "ticket_medio_loja": ticket_medio_loja,
            "ticket_medio_vendedor": ticket_medio_vend,
            "meta_vendedor": meta
        })
        
    return pd.DataFrame(resultados)

In [80]:
df_metas_vendedores = calcular_metas_vendedores(df, ticket_por_hora)

In [84]:
df_metas_vendedores.sort_values(by=['filial_nome', 'meta_vendedor'], ascending=[True, False]).head(20)

,filial_nome,nome_vendedor,vendas_loja,vendas_vendedor,ticket_medio_loja,ticket_medio_vendedor,meta_vendedor
2,DROGARIA JB - F01 - MATRIZ,JOSE ALMERICO TRINDADE FILHO,32618,14766,55.632973,55.632973,0.452695
1,DROGARIA JB - F01 - MATRIZ,BRUNO OLIVEIRA ANDRADE BUCCAREY,32618,7237,55.632973,57.314771,0.228579
3,DROGARIA JB - F01 - MATRIZ,THAIS MENDES DOS SANTOS,32618,5884,55.632973,57.314771,0.185844
0,DROGARIA JB - F01 - MATRIZ,GICELMA SILVA RIBEIRO,32618,5850,55.632973,57.314771,0.184771
7,DROGARIA JB - F01 - MATRIZ,MARIA JOSE DA SILVA,32618,5595,55.632973,57.314771,0.176716
8,DROGARIA JB - F01 - MATRIZ,MARIA EDNALVA DOS SANTOS,32618,4901,55.632973,57.314771,0.154797
5,DROGARIA JB - F01 - MATRIZ,THAYNAR SOUZA SILVA,32618,4453,55.632973,57.314771,0.140647
20,DROGARIA JB - F01 - MATRIZ,FRANCIELE GOMES DE MENEZES,32618,3906,55.632973,57.314771,0.123370
25,DROGARIA JB - F01 - MATRIZ,TAMIRES CAMPOS DE SOUZA,32618,3019,55.632973,57.314771,0.095354
17,DROGARIA JB - F01 - MATRIZ,GABRIELA LARISSA DE OLIVEIRA DALTRO,32618,1559,55.632973,55.632973,0.047796


In [70]:
vendas_vendedores_por_hora = (
    df_90.groupby(['nome_vendedor', 'hora'])
         .agg(n_vendas=('venda_coo', 'nunique'))
         .reset_index()
)

print(vendas_vendedores_por_hora)

                      nome_vendedor  hora  n_vendas
0       ALINE BISPO BIGI DOS SANTOS     7        17
1       ALINE BISPO BIGI DOS SANTOS     8        75
2       ALINE BISPO BIGI DOS SANTOS     9        68
3       ALINE BISPO BIGI DOS SANTOS    10        99
4       ALINE BISPO BIGI DOS SANTOS    11        80
..                              ...   ...       ...
651  WITORIA REGINNA VIEIRA SANTANA    16         1
652  WITORIA REGINNA VIEIRA SANTANA    18         4
653  WITORIA REGINNA VIEIRA SANTANA    19         3
654  WITORIA REGINNA VIEIRA SANTANA    20         6
655  WITORIA REGINNA VIEIRA SANTANA    21         2

[656 rows x 3 columns]


In [24]:
embalagem[embalagem['id']==121838]

,id,produtoid,apresentacao,codigobarras,etiqueta,quantidadeporembalagem,padraofornecedores,embalagemcontidaid,quantidadeembalagem,precoreferencial,...,datahorainclusao,datahoraedicaocodigobarras,usuarioinclusaoid,largura,altura,comprimento,peso,padraointegracoes,quantidadecomercial,unidademedidacomercial
10134,121838,121835,NaN,NaN,355,1,False,NaN,NaN,10.78,...,2024-06-25 16:27:42,2024-07-19 16:23:27,2,NaN,NaN,NaN,NaN,False,NaN,?


In [14]:
item_custo[['codigobarras', 
                  'unidadenegocioid','customedio']].drop_duplicates(subset=['codigobarras', 'unidadenegocioid'])

,codigobarras,unidadenegocioid,customedio
0,7.898278e+12,93723,6.7450
1,7.898278e+12,93724,7.7959
2,7.898278e+12,93725,6.7717
3,7.898278e+12,93732,6.7450
4,7.898278e+12,93733,6.7413
...,...,...,...
51051,7.896095e+12,93731,169.4775
51052,7.896095e+12,93733,166.4700
51053,7.908134e+12,93722,220.2762
51054,7.908134e+12,93732,219.1425


In [17]:
item_custo[(item_custo['embalagemid']==121838)&(item_custo['unidadenegocioid']==93721)]

,embalagemid,unidadenegocioid,customedio,datahora,id,codigobarras
809,121838,93721,3.9475,2025-05-10 12:15:09,121838,NaN


In [19]:
item_custo[item_custo.duplicated(subset=['codigobarras', 'unidadenegocioid'], keep=False)]


,embalagemid,unidadenegocioid,customedio,datahora,id,codigobarras
809,121838,93721,3.9475,2025-05-10 12:15:09,121838,NaN
810,121838,93722,3.8444,2025-05-12 15:39:49,121838,NaN
811,121838,93724,3.8677,2025-05-12 15:27:23,121838,NaN
812,121838,93725,3.8775,2025-04-22 08:11:22,121838,NaN
813,121838,93726,3.8500,2025-05-02 13:48:57,121838,NaN
...,...,...,...,...,...,...
50999,11213281,93733,47.3298,2025-05-08 15:08:22,11213281,NaN
51000,11213283,93733,20.6400,2025-05-08 15:08:22,11213283,NaN
51001,11213285,93733,20.6400,2025-05-08 15:08:22,11213285,NaN
51002,11213287,93733,20.6400,2025-05-08 15:08:22,11213287,NaN


In [15]:
df.merge(item_custo[['codigobarras', 
                  'unidadenegocioid','customedio']].drop_duplicates(subset=['codigobarras', 'unidadenegocioid']), 
        left_on=['embalagem_codigobarras', 
                  'caixa_unidnegocioid'], 
        right_on=['codigobarras', 
                  'unidadenegocioid'],
        how='left')

,filial_codigo,filial_nome,filial_cnpj,filial_cidade,caixa_numero,caixa_unidnegocioid,caixa_descricao,nome_caixa,venda_status,venda_coo,...,produto_movimentacaofracionada,produto_tipomedicamento,caderno_nome,classificacao_nome,classificacao_caminho,classificacao_profundidade,classificacao_paiid,codigobarras,unidadenegocioid,customedio
0,1,DROGARIA JB - F01 - MATRIZ,13145354000176,BOQUIM,1,93733,CAIXA 01,ANA CAROLINE FRAGA OLIVEIRA,C,47662,...,False,A,03 - REFERENCIA - BOQUIM C/IDENTIFICAÇÃO,REFERENCIA OL,PRINCIPAL > PRESCRIÇÃO > PRODUTOS OL > REFEREN...,4,3494179,7.891058e+12,93733.0,13.5000
1,1,DROGARIA JB - F01 - MATRIZ,13145354000176,BOQUIM,1,93733,CAIXA 01,ANA CAROLINE FRAGA OLIVEIRA,F,33698,...,False,B,GEN/SIM - BOQUIM,SIMILAR,PRINCIPAL > INDICAÇÃO > SIMILARES > SIMILAR,4,103400,7.896714e+12,93733.0,3.0656
2,1,DROGARIA JB - F01 - MATRIZ,13145354000176,BOQUIM,1,93733,CAIXA 01,ANA CAROLINE FRAGA OLIVEIRA,F,33699,...,False,D,NaN,PRODUTOS LIBERADOS OL,PRINCIPAL > PRESCRIÇÃO > PRODUTOS OL > PRODUTO...,4,3494179,7.896006e+12,93733.0,114.9343
3,1,DROGARIA JB - F01 - MATRIZ,13145354000176,BOQUIM,1,93733,CAIXA 01,ANA CAROLINE FRAGA OLIVEIRA,F,33700,...,False,A,03 - REFERENCIA - BOQUIM C/IDENTIFICAÇÃO,REFERENCIA OL,PRINCIPAL > PRESCRIÇÃO > PRODUTOS OL > REFEREN...,4,3494179,7.896658e+12,93733.0,134.7000
4,1,DROGARIA JB - F01 - MATRIZ,13145354000176,BOQUIM,1,93733,CAIXA 01,ANA CAROLINE FRAGA OLIVEIRA,F,33700,...,False,A,03 - REFERENCIA - BOQUIM C/IDENTIFICAÇÃO,REFERENCIA FARMACIA POPULAR,PRINCIPAL > PRESCRIÇÃO > MEDICAMENTO REFERENCI...,4,103399,7.891721e+12,93733.0,29.1124
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
329913,3,DROGARIA JB - F03 - AILKA,13145354000680,ESTANCIA,3,93730,CAIXA 03,WITORIA REGINNA VIEIRA SANTANA,G,29905,...,False,D,NaN,VAREJO LIBERADOS,PRINCIPAL > VAREJO > VAREJO LIBERADOS,3,103398,7.893262e+07,93730.0,0.3000
329914,3,DROGARIA JB - F03 - AILKA,13145354000680,ESTANCIA,3,93730,CAIXA 03,WITORIA REGINNA VIEIRA SANTANA,G,29905,...,False,D,NaN,VAREJO LIBERADOS,PRINCIPAL > VAREJO > VAREJO LIBERADOS,3,103398,7.893262e+07,93730.0,0.3000
329915,3,DROGARIA JB - F03 - AILKA,13145354000680,ESTANCIA,3,93730,CAIXA 03,WITORIA REGINNA VIEIRA SANTANA,G,32995,...,False,C,GEN/SIM - ESTANCIA,GENERICO OL,PRINCIPAL > INDICAÇÃO > GENÉRICOS > GENERICO OL,4,103401,7.908021e+12,93730.0,9.4700
329916,3,DROGARIA JB - F03 - AILKA,13145354000680,ESTANCIA,3,93730,CAIXA 03,WITORIA REGINNA VIEIRA SANTANA,G,32995,...,False,D,NaN,HOSPITALAR,PRINCIPAL > VAREJO > HOSPITALAR,3,103398,7.842826e+12,93730.0,0.2869


In [5]:
df.shape

(329918, 68)

In [105]:
mov_filtrado = mov.loc[mov.groupby(['embalagemid', 'unidadenegocioid'])['datahora'].idxmax()]
mov_filtrado.reset_index(drop=True, inplace=True)
mov_filtrado.columns

Index(['codigo_movimentacao', 'descricao_movimentacao', 'embalagemid',
       'unidadenegocioid', 'datahora', 'usuarioid', 'quantidade',
       'estoqueanterior', 'estoque', 'precoreferencial', 'precovenda', 'custo',
       'customedio', 'naoatualizarestoque', 'ultimaalteracaoprecoid',
       'conversaoembalagemid', 'datahorarecalculo', 'customedioanterior',
       'estoqueanteriorproduto', 'custoponderavel', 'origemcustoponderavel'],
      dtype='object')

In [95]:
mov.columns

Index(['codigo_movimentacao', 'descricao_movimentacao', 'embalagemid',
       'unidadenegocioid', 'datahora', 'usuarioid', 'quantidade',
       'estoqueanterior', 'estoque', 'precoreferencial', 'precovenda', 'custo',
       'customedio', 'naoatualizarestoque', 'ultimaalteracaoprecoid',
       'conversaoembalagemid', 'datahorarecalculo', 'customedioanterior',
       'estoqueanteriorproduto', 'custoponderavel', 'origemcustoponderavel'],
      dtype='object')

In [16]:
entregas = pd.read_csv('entregas.csv')
metas = pd.read_csv('metas.csv', sep=';')

In [85]:
df['orcamento_datahora'] = pd.to_datetime(df['orcamento_datahora'], format='%Y-%m-%d %H:%M:%S')

In [86]:
df['orcamento_data'] = pd.to_datetime(df['orcamento_datahora']).dt.date
df['mês/ano'] = df['orcamento_datahora'].dt.to_period('M')
df['hora'] = df['orcamento_datahora'].dt.hour
df['venda_datahoraabertura'] = pd.to_datetime(df['venda_datahoraabertura'], format='%Y-%m-%d %H:%M:%S')
df['venda_datahorafechamento'] = pd.to_datetime(df['venda_datahorafechamento'], format='%Y-%m-%d %H:%M:%S')


In [91]:
df.columns

Index(['filial_codigo', 'filial_nome', 'filial_cnpj', 'filial_cidade',
       'caixa_numero', 'caixa_unidnegocioid', 'caixa_descricao', 'nome_caixa',
       'venda_status', 'venda_coo', 'orcamento_codigo', 'orcamento_tipo',
       'orcamento_status', 'orcamento_datahora', 'orcamento_identificacao',
       'orcamento_usuarioid', 'orcamento_unidadenegocioid',
       'orcamento_formapagamentoid', 'orcamento_pessoaid',
       'orcamento_cpfcnpjconsumidor', 'orcamento_vendaid', 'nome_vendedor',
       'valor_total', 'venda_usuariolibcliente', 'venda_usuariolibpagatrasado',
       'venda_libusuariocancelamento', 'venda_tipodocfiscal',
       'venda_datahoraabertura', 'venda_datahorafechamento', 'item_sequencia',
       'item_status', 'item_tipoaliquota', 'item_quantidade',
       'item_valorunitario', 'item_origemdesconto', 'item_tipodesconto',
       'item_desconto', 'item_valortotal', 'item_cadernoofertaid',
       'item_valordesconto', 'item_movimentacaoestoque',
       'item_diasparavenc

In [87]:
df_filtro = df[['filial_nome', 'nome_vendedor', 'venda_coo',
       'valor_total','venda_datahorafechamento', 
       'item_sequencia','item_quantidade',
       'item_valorunitario', 
       'item_desconto', 'item_valortotal','embalagem_codigobarras',
       'embalagem_descricao', 'mês/ano', 'hora','classificacao_nome',
       'class_painome']]
df_filtro.agg({
    'item_valortotal': 'sum',
    'item_quantidade': 'sum',
    'venda_coo': 'nunique'
})



KeyError: "['class_painome'] not in index"

In [81]:
df_indicacao = df_filtro[df_filtro['class_painome'] == 'INDICAÇÃO']
venda_indicacao = df_indicacao['item_valortotal'].sum()
venda_indicacao

np.float64(2300091.8400000003)

In [80]:
df_filtro['class_painome'].unique()

array(['VAREJO', 'SBB', 'INDICAÇÃO', 'PRESCRIÇÃO', 'QUERODELIVERY',
       'USO CONSUMO E SERVIÇOS'], dtype=object)

In [41]:
# Simulação da data atual para o dashboard
DATA_ATUAL_SIMULADA = datetime(2024, 5, 12).date() # Use o ano correto dos seus dados

def carregar_dados_vendas(caminho_arquivo_vendas="base/vendas2.csv"):
    """Carrega e prepara os dados de vendas."""
    try:
        df = pd.read_csv(caminho_arquivo_vendas)
        # Certifique-se que a coluna de data/hora da venda está no formato datetime
        # Ajuste 'orcamento_datahora' e o formato se necessário
        df['venda_datahorafechamento'] = pd.to_datetime(df['venda_datahorafechamento'], errors='coerce')
        df.dropna(subset=['venda_datahorafechamento'], inplace=True) # Remove linhas onde a conversão falhou
        df['data_venda_apenas'] = df['venda_datahorafechamento'].dt.date
        return df
    except FileNotFoundError:
        print(f"Erro: Arquivo de vendas não encontrado em {caminho_arquivo_vendas}")
        return pd.DataFrame()
    except Exception as e:
        print(f"Erro ao carregar ou processar dados de vendas: {e}")
        return pd.DataFrame()

def calcular_kpi_vendas_loja(df_vendas):
    """Calcula o KPI de venda acumulada no mês e venda do dia anterior."""
    if df_vendas.empty:
        return 0, 0

    mes_atual = DATA_ATUAL_SIMULADA.month
    ano_atual = DATA_ATUAL_SIMULADA.year
    dia_atual_simulado = DATA_ATUAL_SIMULADA.day

    df_vendas['data_venda_apenas'] = pd.to_datetime(df_vendas['data_venda_apenas'], errors='coerce').dt.date

    # Venda acumulada no mês até a DATA_ATUAL_SIMULADA
    vendas_mes_atual_acumulada = df_vendas[
        (df_vendas['data_venda_apenas'].apply(lambda x: x.month) == mes_atual) &
        (df_vendas['data_venda_apenas'].apply(lambda x: x.year) == ano_atual) &
        (df_vendas['data_venda_apenas'].apply(lambda x: x.day) <= dia_atual_simulado)
    ]
    venda_total_mes_acumulada = vendas_mes_atual_acumulada['valor_total'].sum()

    # Venda do dia anterior à DATA_ATUAL_SIMULADA
    data_dia_anterior = DATA_ATUAL_SIMULADA - timedelta(days=1)
    vendas_dia_anterior = df_vendas[df_vendas['data_venda_apenas'] == data_dia_anterior]
    venda_total_dia_anterior = vendas_dia_anterior['item_valortotal'].sum()

    return venda_total_mes_acumulada, venda_total_dia_anterior

# Exemplo de como carregar metas (não usado neste KPI específico, mas para referência futura)
def carregar_dados_metas(caminho_arquivo_metas="base/metas.csv"):
    """Carrega os dados de metas."""
    try:
        df_metas = pd.read_csv(caminho_arquivo_metas, sep=';')
        return df_metas
    except FileNotFoundError:
        print(f"Erro: Arquivo de metas não encontrado em {caminho_arquivo_metas}")
        return pd.DataFrame()
    except Exception as e:
        print(f"Erro ao carregar dados de metas: {e}")
        return pd.DataFrame()


In [42]:
df_vendas_completo = carregar_dados_vendas() # Caminho padrão 'base/vendas2.csv'
venda_acum_mes, venda_dia_anterior = calcular_kpi_vendas_loja(df_vendas_completo)

In [43]:
mes_atual = DATA_ATUAL_SIMULADA.month
ano_atual = DATA_ATUAL_SIMULADA.year
dia_atual_simulado = DATA_ATUAL_SIMULADA.day

In [ ]:
# Venda acumulada no mês até a DATA_ATUAL_SIMULADA
vendas_mes_atual_acumulada = df_vendas_completo[
    (df_vendas_completo['data_venda_apenas'].apply(lambda x: x.month) == mes_atual) &
    (df_vendas_completo['data_venda_apenas'].apply(lambda x: x.year) == ano_atual) &
    (df_vendas_completo['data_venda_apenas'].apply(lambda x: x.day) <= dia_atual_simulado)
]
venda_total_mes_acumulada = vendas_mes_atual_acumulada['valor_total'].sum()

# Venda do dia anterior à DATA_ATUAL_SIMULADA
data_dia_anterior = DATA_ATUAL_SIMULADA - timedelta(days=1)
vendas_dia_anterior = df_vendas_completo[df_vendas_completo['data_venda_apenas'] == data_dia_anterior]
venda_total_dia_anterior = vendas_dia_anterior['item_valortotal'].sum()


In [44]:
df_vendas_completo['data_venda_apenas'].dtypes

dtype('O')

In [32]:
df_vendas_completo[
    (df_vendas_completo['data_venda_apenas'].apply(lambda x: x.month) == mes_atual) &
    (df_vendas_completo['data_venda_apenas'].apply(lambda x: x.year) == ano_atual) &
    (df_vendas_completo['data_venda_apenas'].apply(lambda x: x.day) <= dia_atual_simulado)
]

,filial_codigo,filial_nome,filial_cnpj,filial_cidade,caixa_numero,caixa_unidnegocioid,caixa_descricao,nome_caixa,venda_status,orcamento_codigo,...,produto_principioativoid,produto_usocontinuo,produto_movimentacaofracionada,produto_tipomedicamento,caderno_nome,classificacao_nome,classificacao_caminho,classificacao_profundidade,classificacao_paiid,data_venda_apenas


In [34]:
df_vendas_completo['data_venda_apenas'].drop_duplicates().sort_values(ascending=False).head(10)

111      2025-05-13
105      2025-05-12
2480     2025-05-11
2471     2025-05-10
2361     2025-05-09
2412     2025-05-08
2408     2025-05-07
2370     2025-05-06
2362     2025-05-05
15828    2025-05-04
Name: data_venda_apenas, dtype: object

In [ ]:
df[df['mês/ano']=='2025-04'].groupby(['filial_codigo','filial_nome'])['item_valortotal'].sum().reset_index()

,filial_codigo,filial_nome,valor_total
0,1,DROGARIA JB - F01 - MATRIZ,2338842.43
1,2,DROGARIA JB - F02 - CAMERINO,1194799.75
2,3,DROGARIA JB - F03 - AILKA,1978351.28
3,4,DROGARIA JB - F04 - TOBIAS,2517175.55
4,5,DROGARIA JB - F05 - FILIAL 05,1062831.65
5,6,DROGARIA JB - F06 - JUAREZ,416748.53
6,7,DROGARIA JB - F07 - LAUDELINO,668015.86
7,8,DROGARIA JB - F08 - ROSENDO,867295.62
8,9,DROGARIA JB - F09 - JARDIM VELHO,1637671.95
9,10,DROGARIA JB - F10 - PEDRINHAS,407056.88


In [51]:
classifica = pd.read_csv('base/classificacao.csv')

In [63]:
def extrair_class_painome(caminho):
    if 'QUERODELIVERY' in caminho:
        return 'QUERODELIVERY'
    partes = caminho.split(' > ')
    return partes[1] if len(partes) > 1 else None

df['class_painome'] = df['classificacao_caminho'].apply(extrair_class_painome)

In [64]:
df['class_painome'].unique()

array(['VAREJO', 'SBB', 'INDICAÇÃO', 'PRESCRIÇÃO', 'QUERODELIVERY',
       'USO CONSUMO E SERVIÇOS'], dtype=object)

In [62]:
df['classificacao_caminho'].unique()

array(['PRINCIPAL > VAREJO > LEITES',
       'PRINCIPAL > VAREJO > FRALDA GERIATRICA',
       'PRINCIPAL > SBB > DEPILATORIOS',
       'PRINCIPAL > INDICAÇÃO > GENÉRICOS > GENERICO',
       'PRINCIPAL > INDICAÇÃO > GENÉRICOS > GENERICO PSICOTROPICO',
       'PRINCIPAL > INDICAÇÃO > SIMILARES > SIMILAR',
       'PRINCIPAL > PRESCRIÇÃO > PRODUTOS OL > PRODUTOS LIBERADOS OL',
       'PRINCIPAL > SBB > LINHA CAPILAR',
       'PRINCIPAL > INDICAÇÃO > GENÉRICOS > GENERICO OL',
       'PRINCIPAL > VAREJO > LINHA INFANTIL',
       'PRINCIPAL > PRESCRIÇÃO > MEDICAMENTO REFERENCIA > REFERENCIA',
       'PRINCIPAL > PRESCRIÇÃO > PRODUTOS LIBERADOS > DERMOCOSMETICOS',
       'PRINCIPAL > VAREJO > ABSORV/ALGODAO',
       'PRINCIPAL > VAREJO > VAREJO ORTOPEDICO',
       'PRINCIPAL > VAREJO > FRALDA INFANTIL',
       'PRINCIPAL > PRESCRIÇÃO > PRODUTOS OL > REFERENCIA OL',
       'QUERODELIVERY > ATIVO',
       'PRINCIPAL > PRESCRIÇÃO > PRODUTOS OL > REFERENCIA OL - FRACIONADO',
       'PRINCIPAL > SB

In [20]:
# Converter a string para o tipo datetime.date
data_filtro = datetime.date(2025, 5, 8)


In [23]:
df[(df['venda_coo']==50061)&(df['venda_datahorafechamento'].dt.date==data_filtro)][[
       'valor_total', 'venda_coo',
       'item_sequencia', 'item_quantidade',
       'item_valorunitario', 'item_valortotal', 'embalagem_descricao']]

,valor_total,venda_coo,item_sequencia,item_quantidade,item_valorunitario,item_valortotal,embalagem_descricao
32105,84.16,50061,1,1.0,19.48,18.19,NEOSALDINA C/10 DRG
32106,84.16,50061,2,1.0,96.70,21.99,PREGABALINA 75 MG C/30 CAPS(EUR)
32107,84.16,50061,3,1.0,96.70,21.99,PREGABALINA 75 MG C/30 CAPS(EUR)
32108,84.16,50061,4,1.0,96.70,21.99,PREGABALINA 75 MG C/30 CAPS(EUR)


In [37]:
colunas_agrupamento = [
    'venda_datahorafechamento', 'venda_coo', 'item_valorunitario', 'embalagem_id'
]

df_agrupado = (
    df.groupby(colunas_agrupamento, as_index=False)
      .agg({
          **{col: 'first' for col in df.columns if col not in colunas_agrupamento + ['item_quantidade', 'item_valortotal']},
          'item_quantidade': 'sum',
          'item_valortotal': 'sum'
      })
)

df_agrupado = df_agrupado[df.columns]  # manter a ordem original das colunas

In [7]:
venda08 = df[df['orcamento_data'].astype(str) == '2025-05-08'].groupby(['filial_nome','nome_vendedor']).agg({'item_valortotal':'sum',
                                                                                                   'item_quantidade':'sum',
                                                                                                   'orcamento_codigo':'nunique'}).reset_index()
venda08['ticket_medio'] = venda08['item_valortotal'] / venda08['orcamento_codigo']
venda08[(venda08['filial_nome']=='DROGARIA JB - F01 - MATRIZ')&(venda08['orcamento_codigo']>2)].sort_values(by='ticket_medio', ascending=False)

,filial_nome,nome_vendedor,item_valortotal,item_quantidade,orcamento_codigo,ticket_medio
8,DROGARIA JB - F01 - MATRIZ,THAIS MENDES DOS SANTOS,3963.04,163.0,45,88.067556
6,DROGARIA JB - F01 - MATRIZ,MARIA EDNALVA DOS SANTOS,4135.23,169.0,59,70.088644
5,DROGARIA JB - F01 - MATRIZ,JOSE ALMERICO TRINDADE FILHO,10916.34,399.0,157,69.530828
9,DROGARIA JB - F01 - MATRIZ,THAYNAR SOUZA SILVA,2983.59,144.0,51,58.501765
7,DROGARIA JB - F01 - MATRIZ,MARIA JOSE DA SILVA,3081.69,164.0,53,58.145094
1,DROGARIA JB - F01 - MATRIZ,BRUNO OLIVEIRA ANDRADE BUCCAREY,3282.24,158.0,67,48.988657


In [38]:
df_agrupado[(df_agrupado['venda_datahorafechamento'].dt.date==data_filtro)&(
    df_agrupado['nome_vendedor']=='BRUNO OLIVEIRA ANDRADE BUCCAREY')
   ][['venda_status', 'venda_datahoraabertura', 'venda_datahorafechamento',
      'orcamento_codigo','embalagem_descricao','venda_coo',
       'valor_total', 'item_sequencia',
       'item_status', 'item_quantidade',
       'item_valorunitario', 'item_origemdesconto', 'item_tipodesconto',
       'item_desconto', 'item_valortotal', 
       'item_valordesconto', 'hora'
       ]].to_csv('vendas_08_05_2025.csv', index=False, sep=';', decimal=',')


In [9]:
bruno = pd.read_excel('Venda por item Bruno Oliveira.xls')

In [12]:
compara = bruno_sql[['venda_status','orcamento_codigo', 'embalagem_descricao', 
           'item_quantidade','item_valorunitario', 'item_desconto', 
           'item_valortotal', 'hora','venda_coo']].merge(
               bruno[['HORA', 'Embalagem', 'COO Venda', 'Itens', 'Venda', 'Desconto']], 
       left_on=['embalagem_descricao','hora','venda_coo'], 
       right_on=['Embalagem','HORA','COO Venda'],
       how='left')

In [14]:
compara[compara['HORA'].isna()]

,venda_status,orcamento_codigo,embalagem_descricao,item_quantidade,item_valorunitario,item_desconto,item_valortotal,hora,venda_coo,HORA,Embalagem,COO Venda,Itens,Venda,Desconto
8,F,1242841,VITASANUS AZ MULHER C/60 CAPS SOFTGEL,1.0,64.90,0.00,64.90,7,54575,NaN,NaN,NaN,NaN,NaN,NaN
9,F,1242841,IBUPRIL 600 MG C/20 CPR,1.0,25.31,19.02,6.29,7,54575,NaN,NaN,NaN,NaN,NaN,NaN
11,F,1242841,VITASANUS CALCIO MDK C/60 CAPS,1.0,62.90,0.00,62.90,7,54575,NaN,NaN,NaN,NaN,NaN,NaN
12,F,1242841,EXEVID COQ10 C/30 CAPS,1.0,71.67,21.48,50.19,7,54575,NaN,NaN,NaN,NaN,NaN,NaN
13,F,1242841,LORATADINA 10 MG C/12 CPR(UNI),1.0,20.32,13.43,6.89,7,54575,NaN,NaN,NaN,NaN,NaN,NaN
14,F,1242841,VITASANUS MAGNESIO TREONATO C/60 CAPS,1.0,64.90,0.00,64.90,7,54575,NaN,NaN,NaN,NaN,NaN,NaN
15,F,1242846,DIURIX 25 MG C/30 CPR,1.0,4.95,3.15,1.80,7,50016,NaN,NaN,NaN,NaN,NaN,NaN
18,F,1242846,GLIBENCLAMIDA 5 MG C/30 CPR(CIM),1.0,16.40,14.00,2.40,7,50016,NaN,NaN,NaN,NaN,NaN,NaN
19,F,1242846,LOSARTANA POTASSICA 50 MG C/30 CPR(TEU),1.0,10.38,4.98,5.40,7,50016,NaN,NaN,NaN,NaN,NaN,NaN
20,F,1242846,CLOR DE METFORMINA 500 MG C/30 CPR(PRA),1.0,8.58,4.38,4.20,7,50016,NaN,NaN,NaN,NaN,NaN,NaN


In [74]:
bruno[['HORA', 'Embalagem', 'COO Venda', 'Itens', 'Venda', 'Desconto']].merge(bruno_sql[['venda_status', 
       'orcamento_codigo', 'embalagem_descricao', 'item_quantidade','item_valorunitario', 'item_desconto', 'item_valortotal', 'hora','venda_coo']], 
       left_on=['Embalagem','HORA','COO Venda'], right_on=['embalagem_descricao','hora','venda_coo'],
       how='outer')

,HORA,Embalagem,COO Venda,Itens,Venda,Desconto,venda_status,orcamento_codigo,embalagem_descricao,item_quantidade,item_valorunitario,item_desconto,item_valortotal,hora,venda_coo
0,15.0,ABS SEMPRE LIVRE ADAPT C/8 C/ABAS SUAVE,54668.0,1.0,4.08,1.79,F,1243131.0,ABS SEMPRE LIVRE ADAPT C/8 C/ABAS SUAVE,1.0,5.87,1.79,4.08,15.0,54668.0
1,7.0,ABS SEMPRE LIVRE ADAPT SUAVE C/A C/32 UND,54562.0,1.0,16.32,0.00,F,1242831.0,ABS SEMPRE LIVRE ADAPT SUAVE C/A C/32 UND,1.0,16.32,0.00,16.32,7.0,54562.0
2,9.0,ABS SEMPRE LIVRE ADAPT SUAVE C/A C/32 UND,50051.0,1.0,16.32,0.00,F,1242916.0,ABS SEMPRE LIVRE ADAPT SUAVE C/A C/32 UND,1.0,16.32,0.00,16.32,9.0,50051.0
3,7.0,ABS SEMPRE LIVRE ESPECIAL C/8 C/ABAS ADAPT,54562.0,1.0,4.08,0.54,F,1242831.0,ABS SEMPRE LIVRE ESPECIAL C/8 C/ABAS ADAPT,1.0,4.62,0.54,4.08,7.0,54562.0
4,9.0,ABS SEMPRE LIVRE ESPECIAL C/8 C/ABAS ADAPT,50050.0,1.0,4.08,0.54,F,1242917.0,ABS SEMPRE LIVRE ESPECIAL C/8 C/ABAS ADAPT,1.0,4.62,0.54,4.08,9.0,50050.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
178,8.0,VITASANUS AZ MULHER C/60 CAPS SOFTGEL,54575.0,1.0,64.90,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
179,NaN,NaN,NaN,NaN,NaN,NaN,F,1242841.0,VITASANUS CALCIO MDK C/60 CAPS,1.0,62.90,0.00,62.90,7.0,54575.0
180,8.0,VITASANUS CALCIO MDK C/60 CAPS,54575.0,1.0,62.90,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
181,NaN,NaN,NaN,NaN,NaN,NaN,F,1242841.0,VITASANUS MAGNESIO TREONATO C/60 CAPS,1.0,64.90,0.00,64.90,7.0,54575.0


In [122]:
classificacao = df[df['orcamento_data'].astype(str) == '2025-05-08'].groupby(['filial_nome','classificacao_nome']).agg({'item_valortotal':'sum',
                                                                                                    'item_quantidade':'sum',
                                                                                                    'orcamento_codigo':'nunique'}).reset_index()
classificacao['ticket_medio'] = classificacao['item_valortotal'] / classificacao['orcamento_codigo']


In [137]:
# Para cada combinação de embalagemid e unidadenegocioid, pegar a linha com a datahora mais recente
mov_filtrado = mov.loc[mov.groupby(['embalagemid', 'unidadenegocioid'])['datahora'].idxmax()]

# Resetar o índice para organizar o DataFrame
mov_filtrado = mov_filtrado.reset_index(drop=True)
mov_filtrado[['embalagemid', 'unidadenegocioid', 'datahora', 'customedio']].head(10)

,embalagemid,unidadenegocioid,datahora,customedio
0,103504,93723,2025-04-05 18:24:43,6.7450
1,103504,93724,2025-05-10 11:28:45,7.7959
2,103504,93725,2025-05-09 18:12:14,6.7717
3,103504,93732,2025-04-19 12:20:09,6.7450
4,103504,93733,2025-05-10 17:29:07,6.7413
5,103561,93731,2025-04-29 15:20:37,3.7400
6,103561,93733,2025-05-10 17:59:03,3.4600
7,103711,93722,2025-04-28 17:44:17,3.8975
8,103711,93725,2025-04-15 16:11:06,3.9225
9,103711,93729,2025-04-15 15:38:27,3.9125


In [ ]:
df = df.merge(mov_filtrado[['embalagemid', 'unidadenegocioid', 'customedio']], left_on=['caixa_unidnegocioid','embalagem_id'], right_on=['unidadenegocioid','embalagemid'], how='left')

In [147]:
df['customedio_total'] = df['customedio'] * df['item_quantidade']

In [150]:
venda_dia = df.groupby(['orcamento_data']).agg({'item_valortotal':'sum',
                                        'customedio_total':'sum',
                                        }).reset_index()
venda_dia['lucro_bruto'] = venda_dia['item_valortotal'] - venda_dia['customedio_total']
venda_dia['margem_lucro'] = venda_dia['lucro_bruto'] / venda_dia['item_valortotal']
venda_dia

,orcamento_data,item_valortotal,customedio_total,lucro_bruto,margem_lucro
0,2025-04-01,177086.98,128120.7009,48966.2791,0.276510
1,2025-04-02,170951.48,122935.6919,48015.7881,0.280874
2,2025-04-03,174463.10,125688.9165,48774.1835,0.279567
3,2025-04-04,170647.70,123822.2300,46825.4700,0.274398
4,2025-04-05,147269.72,106877.6595,40392.0605,0.274273
5,2025-04-06,61430.50,44397.0960,17033.4040,0.277279
6,2025-04-07,217687.80,157947.7301,59740.0699,0.274430
7,2025-04-08,179472.97,129961.6978,49511.2722,0.275870
8,2025-04-09,176245.77,128254.6443,47991.1257,0.272297
9,2025-04-10,162917.62,116284.4733,46633.1467,0.286238


In [51]:
mov['datahora'] = pd.to_datetime(mov['datahora'], format='%Y-%m-%d %H:%M:%S')

In [53]:
mov['datahora'].min()

Timestamp('2025-04-01 01:44:10')

In [58]:
movimentacao = mov[['descricao_movimentacao', 'embalagemid',
       'unidadenegocioid', 'datahora', 'usuarioid', 'quantidade',
       'estoqueanterior', 'estoque', 'precoreferencial', 'precovenda', 'custo',
       'customedio']].merge(embalagem[['id', 'apresentacao', 'codigobarras', 'etiqueta',
       'quantidadeporembalagem', 
       'quantidadeembalagem', 'precoreferencial', 'markup', 'precovenda',
       'descricao', 'precovendavariavel', 'margemlucrovenda']], 
       left_on='embalagemid', right_on='id', how='left'
       ).copy()

In [71]:
lojas = df[['filial_codigo','filial_nome','filial_cidade','caixa_unidnegocioid']].drop_duplicates()

In [74]:
movimentacao = movimentacao.merge(lojas[['filial_nome','filial_cidade','caixa_unidnegocioid']], left_on='unidadenegocioid', right_on='caixa_unidnegocioid', how='left')

In [77]:
movimentacao[(movimentacao['estoque']==0)&(movimentacao['filial_nome']=='DROGARIA JB - F01 - MATRIZ')&(movimentacao['datahora'].dt.month == 4)].groupby(['filial_nome','codigobarras','descricao'])['codigobarras'].count().reset_index(name='quantidade').sort_values(by='quantidade', ascending=False).head(10)

,filial_nome,codigobarras,descricao,quantidade
190,DROGARIA JB - F01 - MATRIZ,7.891058e+12,DIPIRONA MONO 500 MG C/10 CPR(MED) CX C/10,13
478,DROGARIA JB - F01 - MATRIZ,7.895858e+12,MAGNESIA BISURADA C/10 PAST MENTA CX C/ 20,13
0,DROGARIA JB - F01 - MATRIZ,1.089220e+05,TESTE DE GLICOSE C/1 UND,6
1621,DROGARIA JB - F01 - MATRIZ,7.899941e+12,CARB UP BLACK 30G GUARANA/ACAI CX C/ 10,5
297,DROGARIA JB - F01 - MATRIZ,7.891317e+12,TAMISA CARTELA 30 MG C/21 CPR CX C/ 3,5
1512,DROGARIA JB - F01 - MATRIZ,7.898947e+12,LUVA PROCEDIMENTO TAM M TOP QUALITY CX C/ 100,5
761,DROGARIA JB - F01 - MATRIZ,7.896062e+12,FRALDA BABYSEC SHORTINHO HIPER TAM XG C/42 UND,4
1274,DROGARIA JB - F01 - MATRIZ,7.897931e+12,SOAPEX 1% SAB LIQ 120 ML,4
1653,DROGARIA JB - F01 - MATRIZ,7.908134e+12,GYNPRO 100 MG C/30 CAPS,4
806,DROGARIA JB - F01 - MATRIZ,7.896095e+12,LACTO PURGA C/6 CPR CX C/25,4


In [66]:
movimentacao.groupby(['codigobarras','descricao'])['estoque'].mean().reset_index(name='quantidade').sort_values(by='quantidade', ascending=False).head(10)

,codigobarras,descricao,quantidade
5018,7.896112e+12,LOSARTANA POTASSICA 50 MG C/30 CPR(TEU),1940.187483
2864,7.895296e+12,DIPIRONA MONO 500 MG C/10 CPR(NOQ),1171.327511
8384,7.898401e+12,"COMPRESSA DE GAZE 7,5 X 7,5 HEMOGAZE",692.753275
5055,7.896112e+12,CLOR DE METFORMINA 850 MG C/30 CPR(TEU),675.882963
2465,7.891721e+12,GLIFAGE XR 500 MG C/30 CPR,544.740262
6973,7.896714e+12,DORALGINA C/4 DRG,485.729611
9018,7.898668e+12,"COMPRESSA DE GAZE 7,5 X 7,5 13F LIVIA",463.964286
640,7.842826e+12,SERINGA SR 5ML S/AGULHA LUER SLIP,410.466667
1237,7.891058e+12,DIPIRONA MONO 500 MG C/10 CPR(MED),383.977641
4743,7.896095e+12,EPOCLER UNIDADE C/1 UND CX C/60,377.974684


In [86]:
entregas['datahorainicial'] = pd.to_datetime(entregas['datahorainicial'], format='%Y-%m-%d %H:%M:%S')
entregas['datahoraprogramada'] = pd.to_datetime(entregas['datahoraprogramada'], format='%Y-%m-%d %H:%M:%S')
entregas['datahorafinal'] = pd.to_datetime(entregas['datahorafinal'], format='%Y-%m-%d %H:%M:%S')
entregas['cumprimento_prazo'] = np.where(entregas['datahorafinal'] <= entregas['datahoraprogramada'], 1, 0)
entregas['tempo_entrega'] = (entregas['datahorafinal'] - entregas['datahorainicial']).dt.total_seconds() / 3600
entregas['mês/ano'] = entregas['datahorainicial'].dt.to_period('M')


In [90]:
# Agrupar por 'mês/ano' e calcular as métricas
resultado = entregas.groupby('mês/ano').agg(
    total_orcamentoid=('orcamentoid', 'count'),  # Número total de orcamentoid
    total_cumprimento_prazo_1=('cumprimento_prazo', lambda x: (x == 1).sum()),  # Total com cumprimento_prazo = 1
    total_cumprimento_prazo_0=('cumprimento_prazo', lambda x: (x == 0).sum()),  # Total com cumprimento_prazo = 0
    tempo_medio_entrega=('tempo_entrega', 'mean')  # Tempo médio de entrega
).reset_index()
resultado['proporcao_cumprimento'] = resultado['total_cumprimento_prazo_1'] / resultado['total_orcamentoid']
resultado

,mês/ano,total_orcamentoid,total_cumprimento_prazo_1,total_cumprimento_prazo_0,tempo_medio_entrega,proporcao_cumprimento
0,2024-07,2114,1025,1089,4.958332,0.484863
1,2024-08,3227,1502,1725,6.217868,0.465448
2,2024-09,3296,1679,1617,4.405402,0.509405
3,2024-10,3389,1524,1865,4.917339,0.449690
4,2024-11,2884,1308,1576,3.441684,0.453537
5,2024-12,2882,1325,1557,2.720234,0.459750
6,2025-01,3005,1387,1618,2.373910,0.461564
7,2025-02,2861,1225,1636,3.983817,0.428172
8,2025-03,3269,1342,1927,5.888144,0.410523
9,2025-04,2825,1173,1652,2.772466,0.415221


In [94]:
entregas[(entregas['cumprimento_prazo']==0)&(entregas['tempo_entrega']<=24)][['datahorainicial','datahorafinal','datahoraprogramada','tempo_entrega']].sort_values(by='tempo_entrega', ascending=False).head(10)

,datahorainicial,datahorafinal,datahoraprogramada,tempo_entrega
30719,2025-05-09 09:38:04,2025-05-10 09:37:49,2025-05-09 10:38:04,23.995833
28844,2025-04-20 16:22:02,2025-04-21 16:21:07,2025-04-20 17:22:02,23.984722
4944,2024-08-28 14:31:02,2024-08-29 14:28:13,2024-08-28 15:31:02,23.953056
25283,2025-03-13 18:51:46,2025-03-14 18:47:17,2025-03-13 19:51:46,23.925278
28767,2025-04-23 11:43:35,2025-04-24 11:37:57,2025-04-23 12:43:35,23.906111
15161,2024-12-05 12:56:49,2024-12-06 12:51:00,2024-12-05 13:56:49,23.903056
3981,2024-08-18 12:37:30,2024-08-19 12:29:19,2024-08-18 13:37:30,23.863611
3298,2024-08-12 09:00:54,2024-08-13 08:50:45,2024-08-12 10:00:54,23.830833
1963,2024-07-30 12:27:46,2024-07-31 12:17:28,2024-07-30 13:27:46,23.828333
4947,2024-09-01 09:37:54,2024-09-02 09:24:44,2024-09-01 10:37:54,23.780556
